# 🎬 IMDb Top 1000 Movies — Exploratory Data Analysis

**Goal:** Answer 5 real questions about the best movies ever made using pandas and numpy.

**Dataset:** IMDb Top 1000 Movies (Kaggle)

**Questions we'll answer:**
1. Which genres have the highest average IMDb rating?
2. Who are the top 10 directors by number of movies?
3. Is there a correlation between votes and rating?
4. How have average ratings changed across decades?
5. Which movies have missing revenue — is there a pattern?

## 📦 Imports

In [ ]:
import pandas as pd
import numpy as np

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

## 📥 Load Data

In [ ]:
df = pd.read_csv("imdb_top_1000.csv")
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

## 🔍 Data Audit

In [ ]:
print("=== NULL COUNTS ===")
print(df.isnull().sum())
print("\n=== DATA TYPES ===")
print(df.dtypes)
print("\n=== BASIC STATS ===")
df[['IMDB_Rating', 'No_of_Votes', 'Meta_score']].describe()

## 🧹 Data Cleaning

In [ ]:
df['Released_Year'] = pd.to_numeric(df['Released_Year'], errors='coerce')
df['Runtime'] = df['Runtime'].str.replace(" min", "").astype(int)
df['Gross'] = df['Gross'].str.replace(",", "").astype(float)
df['Decade'] = (df['Released_Year'] // 10 * 10).astype("Int64")
print("Cleaning Done ✅")
print(df[['Series_Title', 'Runtime', 'Gross', 'Decade']].head())

---
## ❓ Question 1 — Which genres have the highest average IMDb rating?

In [ ]:
genre_df = df['Genre'].str.split(", ").explode()
genre_rating = pd.DataFrame({
    "Genre" : genre_df,
    "IMDB_Rating" : df.loc[genre_df.index, "IMDB_Rating"]
})
genre_avg = genre_rating.groupby("Genre")["IMDB_Rating"].mean().sort_values(ascending=False)
print(genre_avg)
print("\nTop 3 genres:", genre_avg.head(3).index.tolist())

---
## ❓ Question 2 — Who are the top 10 directors by number of movies?

In [ ]:
top_directors = df["Director"].value_counts().head(10)
print("Top 10 Directors by Movie Count:")
print(top_directors)
director_avg_rating = df.groupby("Director")["IMDB_Rating"].mean()
top_director_names = top_directors.index.tolist()
print("\nTheir average IMDb ratings:")
print(director_avg_rating.loc[top_director_names].round(2))

---
## ❓ Question 3 — Is there a correlation between number of votes and IMDb rating?

In [ ]:
correlation = df['No_of_Votes'].corr(df['IMDB_Rating'])
print(f"Correlation between Votes and Rating: {correlation:.4f}")
df["Vote_Range"] = pd.cut(df["No_of_Votes"],
                          bins=[0, 50000, 200000, 500000, 2500000],
                          labels=["Low", "Medium", "High", "Very High"])
vote_rating = df.groupby("Vote_Range", observed=True)["IMDB_Rating"].mean().round(3)
print("\nAverage rating by vote range:")
print(vote_rating)
print("\nMovie count per range:")
print(df["Vote_Range"].value_counts().sort_index())

---
## ❓ Question 4 — How have average ratings changed across decades?

In [ ]:
decade_stats = df.groupby("Decade", observed=True).agg(
    average_rating = ("IMDB_Rating", "mean"),
    movie_count = ("Series_Title","count")
).round(2)
print(decade_stats)

---
## ❓ Question 5 — Which movies have missing revenue — is there a pattern?

In [ ]:
df["Has_Gross"] = df["Gross"].isnull()
print("=== Average rating: movies WITH vs WITHOUT gross data ===")
print(df.groupby("Has_Gross")["IMDB_Rating"].mean().round(3))
print("\n=== Missing gross by decade ===")
missing_by_decade = df[df["Has_Gross"] == True].groupby("Decade", observed=True)["Series_Title"].count()
print(missing_by_decade)
print("\n=== Missing gross % by decade ===")
total_by_decade = df.groupby("Decade")["Series_Title"].count()
missing_pct = (missing_by_decade / total_by_decade * 100).round(1)
print(missing_pct)

---
## 📋 Key Findings

1. **Genre** — War, Western and Film-Noir have the highest average ratings. Popular genres like Action and Comedy rank near the bottom.

2. **Directors** — Christopher Nolan has the highest average rating (8.46) among top directors. Alfred Hitchcock leads in volume (14 movies) but has the lowest average among the top 10.

3. **Votes vs Rating** — Moderate positive correlation (0.495). Movies with very high vote counts rate significantly higher (8.178) than the rest (~7.90).

4. **Decades** — Older decades have fewer but higher-rated movies. Time filters out bad films. 2000s and 2010s dominate in count but have the lowest average ratings.

5. **Missing Revenue** — Missing gross data is heavily concentrated in pre-1980s movies (30-45% missing) due to inconsistent historical record-keeping — not movie quality.

---

## 💾 Save Cleaned Dataset

In [ ]:
df.drop(columns=["Has_Gross", "Vote_Range"], inplace=True)
df.to_csv("imdb_top_1000_cleaned.csv", index=False)
print("\nCleaned dataset saved ✅")
print(f"Final shape: {df.shape}")